<a href="https://colab.research.google.com/github/harsha-suribhatla/promptengineeringassignment/blob/main/Case1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Case 1 Prompts:
Write Python code for Google Colab that uses from google.colab import ai and ai.generate_text() to build a 4-step customer support prompt chain. Each step's output must be passed into the next step's prompt. Steps: 1) classify the ticket as JSON, 2) generate clarifying questions, 3) draft a solution using the ticket, the classification, and simulated customer answers, 4) apply escalation rules and return JSON. Parse JSON outputs with json.loads inside try/except. Print a clear header and the output for each step. Test with 2 sample tickets: one simple billing issue and one angry fraud complaint.

You are a customer support triage agent. Classify the ticket below. Return ONLY valid JSON with keys: "category" (one of: Billing, Technical, Account Access, Shipping, Other), "urgency" (Low/Medium/High), "sentiment" (Calm/Frustrated/Angry), "summary" (one sentence, max 20 words). Do not add explanations.
Ticket: {ticket}

You are a support agent. Given the ticket and its classification, list up to 3 clarifying questions needed to resolve it. Be polite and specific. Never ask for passwords, full card numbers, or SSNs. Return a numbered list only.
Ticket: {ticket}
Classification: {step1_output}

You are a friendly, professional support agent. Using the ticket, the classification, and the customer's answers, write a reply under 120 words. Start with one empathetic sentence, then give numbered resolution steps. Do not promise refunds or credits; say "our billing team will review" instead.
Ticket: {ticket}
Classification: {step1_output}
Questions asked: {step2_output}
Customer answers: {simulated_answers}

Decide whether this case needs a human. Escalate if ANY of these are true: urgency is High, sentiment is Angry, the ticket mentions fraud, legal action, or a data breach, or the proposed solution cannot resolve the issue. Return ONLY JSON: {"escalate": true/false, "team": "Tier 2 / Fraud / Billing / None", "reason": "<15 words"}.
Classification: {step1_output}
Proposed solution: {step3_output}


```



In [ ]:
from google.colab import ai
import json

def generate_text_with_json_parse(prompt_text, error_message="Could not parse JSON."):
    """Helper function to generate text and safely parse JSON if applicable."""
    response_text = ai.generate_text(prompt=prompt_text)
    try:
        # Attempt to parse as JSON. If it's not JSON, it will raise an error.
        parsed_json = json.loads(response_text)
        return parsed_json, response_text
    except json.JSONDecodeError:
        # If not JSON, return the raw text and None for parsed_json
        return None, response_text

In [ ]:
def run_customer_support_chain(ticket_description):
    print(f"\n{'='*50}\nProcessing Ticket:\n{ticket_description}\n{'='*50}")

    # --- Step 1: Classify the ticket as JSON ---
    step1_prompt = f"""
    Classify the following customer support ticket into categories (e.g., 'Billing', 'Technical', 'Product', 'Fraud', 'Account') and determine the sentiment (e.g., 'Positive', 'Neutral', 'Negative', 'Angry'). Return the classification as a JSON object.

    Ticket: {ticket_description}

    JSON Classification:
    """
    print("\n--- Step 1: Ticket Classification ---")
    classification_json, classification_raw = generate_text_with_json_parse(step1_prompt)
    if classification_json:
        print(f"Classification (Parsed JSON): {json.dumps(classification_json, indent=2)}")
    else:
        print(f"Classification (Raw Text): {classification_raw}")
        print("Warning: Step 1 did not return valid JSON. Proceeding with raw text if possible.")

    # Prepare for next step, ensuring classification data is a string for prompt concatenation
    classification_info = json.dumps(classification_json) if classification_json else classification_raw

    # --- Step 2: Generate clarifying questions ---
    step2_prompt = f"""
    Based on the following customer support ticket and its classification, generate 2-3 clarifying questions that a customer support agent should ask to gather more information. Return only the questions as a bulleted list.

    Ticket: {ticket_description}
    Classification: {classification_info}

    Clarifying Questions:
    """
    print("\n--- Step 2: Clarifying Questions ---")
    _, clarifying_questions = generate_text_with_json_parse(step2_prompt)
    print(clarifying_questions)

    # --- Simulate customer answers (for demonstration purposes) ---
    # In a real scenario, these would come from actual customer interaction
    simulated_answers = {
        "Billing": "I was charged twice for my last month's subscription. My account is user@example.com.",
        "Fraud": "I noticed several unauthorized transactions on my credit card associated with this account. The charges started last night.",
        "Default": "The issue started yesterday."
    }
    ticket_category = classification_json.get('categories', ['Default'])[0] if classification_json and 'categories' in classification_json else 'Default'
    customer_simulated_answer = simulated_answers.get(ticket_category, simulated_answers['Default'])
    print(f"\n(Simulated Customer Answer: {customer_simulated_answer})")

    # --- Step 3: Draft a solution ---
    step3_prompt = f"""
    Given the original customer support ticket, its classification, and the simulated customer answers, draft a polite and helpful initial solution or next steps for the customer.

    Ticket: {ticket_description}
    Classification: {classification_info}
    Customer Answer: {customer_simulated_answer}

    Drafted Solution:
    """
    print("\n--- Step 3: Draft Solution ---")
    _, drafted_solution = generate_text_with_json_parse(step3_prompt)
    print(drafted_solution)

    # --- Step 4: Apply escalation rules and return JSON ---
    step4_prompt = f"""
    Considering the original ticket, its classification, and the drafted solution, apply the following escalation rules and return the result as a JSON object:
    - If the sentiment is 'Angry' or a category includes 'Fraud', escalate to a 'Specialist Team'.
    - If the category is 'Billing' and the solution requires a refund, escalate to 'Finance Department'.
    - Otherwise, mark as 'Standard Resolution'.

    Include the escalation status, a reason, and suggested next agent action. If no escalation, state 'None'.

    Ticket: {ticket_description}
    Classification: {classification_info}
    Drafted Solution: {drafted_solution}

    JSON Escalation Decision:
    """
    print("\n--- Step 4: Escalation Rules ---")
    escalation_json, escalation_raw = generate_text_with_json_parse(step4_prompt)
    if escalation_json:
        print(f"Escalation Decision (Parsed JSON): {json.dumps(escalation_json, indent=2)}")
    else:
        print(f"Escalation Decision (Raw Text): {escalation_raw}")
        print("Warning: Step 4 did not return valid JSON.")

    print(f"\n{'='*50}\nTicket Processing Complete\n{'='*50}")

### Test Cases

In [ ]:
# Sample Ticket 1: Simple Billing Issue
sample_ticket_1 = "I was charged twice for my subscription this month. This is unacceptable! I expect a refund for the extra charge immediately. My account number is 12345."
run_customer_support_chain(sample_ticket_1)

# Sample Ticket 2: Angry Fraud Complaint
sample_ticket_2 = "I am absolutely furious! Someone has been using my card for purchases I didn't make. This is fraud! I demand immediate action and a full investigation. My card number associated with the account ends in 5678."
run_customer_support_chain(sample_ticket_2)

In [ ]:
def classify_ticket_with_prompt(ticket):
    prompt = f"""
    You are a customer support triage agent. Classify the ticket below. Return ONLY valid JSON with keys: "category" (one of: Billing, Technical, Account Access, Shipping, Other), "urgency" (Low/Medium/High), "sentiment" (Calm/Frustrated/Angry), "summary" (one sentence, max 20 words). Do not add explanations.
    Ticket: {ticket}
    """
    print(f"\n{'='*50}\nClassifying Ticket: {ticket}\n{'='*50}")
    classified_json, classified_raw = generate_text_with_json_parse(prompt)
    if classified_json:
        print(f"Classification (Parsed JSON): {json.dumps(classified_json, indent=2)}")
        return classified_json
    else:
        print(f"Classification (Raw Text): {classified_raw}")
        print("Warning: Classification did not return valid JSON.")
        return None

In [ ]:
def generate_clarifying_questions(ticket, classification):
    prompt = f"""
    You are a support agent. Given the ticket and its classification, list up to 3 clarifying questions needed to resolve it. Be polite and specific. Never ask for passwords, full card numbers, or SSNs. Return a numbered list only.
    Ticket: {ticket}
    Classification: {classification}
    """
    print(f"\n{'='*50}\nGenerating Clarifying Questions for Ticket:\n{ticket}\n{'='*50}")
    _, questions = generate_text_with_json_parse(prompt)
    print(questions)
    return questions

In [ ]:
from google.colab import ai
import json

def generate_text_with_json_parse(prompt_text, error_message="Could not parse JSON."):
    """Helper function to generate text and safely parse JSON if applicable."""
    response_text = ai.generate_text(prompt=prompt_text)
    try:
        # Attempt to parse as JSON. If it's not JSON, it will raise an error.
        parsed_json = json.loads(response_text)
        return parsed_json, response_text
    except json.JSONDecodeError:
        # If not JSON, return the raw text and None for parsed_json
        return None, response_text

def draft_solution(ticket, classification, questions_asked, customer_answers):
    prompt = f"""
    You are a friendly, professional support agent. Using the ticket, the classification, and the customer's answers, write a reply under 120 words. Start with one empathetic sentence, then give numbered resolution steps. Do not promise refunds or credits; say "our billing team will review" instead.
    Ticket: {ticket}
    Classification: {classification}
    Questions asked: {questions_asked}
    Customer answers: {customer_answers}
    """
    print(f"\n{'='*50}\nDrafting Solution for Ticket:\n{ticket}\n{'='*50}")
    _, solution_draft = generate_text_with_json_parse(prompt)
    print(solution_draft)
    return solution_draft

In [ ]:
def apply_escalation_rules(classification, proposed_solution):
    prompt = f"""
    Decide whether this case needs a human. Escalate if ANY of these are true: urgency is High, sentiment is Angry, the ticket mentions fraud, legal action, or a data breach, or the proposed solution cannot resolve the issue. Return ONLY JSON: {{"escalate": true/false, "team": "Tier 2 / Fraud / Billing / None", "reason": "<15 words"}}.
    Classification: {classification}
    Proposed solution: {proposed_solution}
    """
    print(f"\n{'='*50}\nApplying Escalation Rules with Classification:\n{classification}\nAnd Proposed Solution:\n{proposed_solution}\n{'='*50}")
    escalation_json, escalation_raw = generate_text_with_json_parse(prompt)
    if escalation_json:
        print(f"Escalation Decision (Parsed JSON): {json.dumps(escalation_json, indent=2)}")
        return escalation_json
    else:
        print(f"Escalation Decision (Raw Text): {escalation_raw}")
        print("Warning: Escalation decision did not return valid JSON.")
        return None

In [ ]:
# Example usage of the new classification function
sample_ticket = "My internet is not working at all. I've tried restarting the router multiple times. This is very frustrating as I work from home!"
classified_ticket = classify_ticket_with_prompt(sample_ticket)

In [ ]:
# Example usage of the clarifying questions function
sample_ticket_for_questions = "My new product arrived damaged. The box was completely crushed. I need a replacement right away."
sample_classification_for_questions = {"category": "Shipping", "urgency": "High", "sentiment": "Frustrated", "summary": "New product arrived damaged; customer needs replacement."}

clarifying_questions = generate_clarifying_questions(sample_ticket_for_questions, sample_classification_for_questions)

In [ ]:
# Example usage of the draft solution function
sample_ticket_for_solution = "My new product arrived damaged. The box was completely crushed. I need a replacement right away."
sample_classification_for_solution = {"category": "Shipping", "urgency": "High", "sentiment": "Frustrated", "summary": "New product arrived damaged; customer needs replacement."}
sample_questions_asked = "1. Can you confirm the order number?\n2. Please provide photos of the damaged product and packaging."
simulated_customer_answers_for_solution = "My order number is 12345. I have attached photos of the crushed box and broken item."

drafted_reply = draft_solution(sample_ticket_for_solution, sample_classification_for_solution, sample_questions_asked, simulated_customer_answers_for_solution)


Drafting Solution for Ticket:
My new product arrived damaged. The box was completely crushed. I need a replacement right away.
I am incredibly sorry to hear that your new product arrived damaged, and I completely understand how frustrating it is to receive your package in this condition. 

Here is how we will resolve this for you:

1. **Process Replacement:** I have used your order number 12345 and the photos you provided to initiate a priority replacement order immediately.
2. **Send Tracking Details:** You will receive an email with a new tracking link as soon as your replacement leaves our warehouse.
3. **Account Review:** Our billing team will review your order to ensure no additional shipping or replacement fees are charged to your account.


In [ ]:
# Example usage of the apply_escalation_rules function
sample_classification_for_escalation = {"category": "Shipping", "urgency": "High", "sentiment": "Frustrated", "summary": "New product arrived damaged; customer needs replacement."}
sample_proposed_solution = "We apologize for the damaged product. To resolve this, please return the item using the pre-paid label, and upon receipt, we will ship a replacement. Our team will ensure this doesn't happen again."

escalation_decision = apply_escalation_rules(sample_classification_for_escalation, sample_proposed_solution)

# Another example for a fraud case
f,s = generate_text_with_json_parse("Classify the following customer support ticket into categories (e.g., 'Billing', 'Technical', 'Product', 'Fraud', 'Account') and determine the sentiment (e.g., 'Positive', 'Neutral', 'Negative', 'Angry'). Return the classification as a JSON object.\n\nTicket: I was charged twice for my subscription this month. This is unacceptable! I expect a refund for the extra charge immediately. My account number is 12345.\n\nJSON Classification:")
fraud_classification = {'category': 'Fraud', 'urgency': 'High', 'sentiment': 'Angry', 'summary': 'Customer reports unauthorized charges and demands immediate action.'}
fraud_solution = "I understand your frustration with the unauthorized charges. Our billing team will review this issue immediately and investigate the transactions. Please monitor your account for updates."

escalation_decision_fraud = apply_escalation_rules(fraud_classification, fraud_solution)


Applying Escalation Rules with Classification:
{'category': 'Shipping', 'urgency': 'High', 'sentiment': 'Frustrated', 'summary': 'New product arrived damaged; customer needs replacement.'}
And Proposed Solution:
We apologize for the damaged product. To resolve this, please return the item using the pre-paid label, and upon receipt, we will ship a replacement. Our team will ensure this doesn't happen again.
Escalation Decision (Raw Text): ```json
{"escalate": true, "team": "Tier 2", "reason": "Urgency is High"}
```

Applying Escalation Rules with Classification:
{'category': 'Fraud', 'urgency': 'High', 'sentiment': 'Angry', 'summary': 'Customer reports unauthorized charges and demands immediate action.'}
And Proposed Solution:
I understand your frustration with the unauthorized charges. Our billing team will review this issue immediately and investigate the transactions. Please monitor your account for updates.
Escalation Decision (Parsed JSON): {
  "escalate": true,
  "team": "Fraud",